In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.8 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import shutil
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO
import matplotlib.pyplot as plt
import torch

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
base_path = "/content/drive/My Drive/new grad data"
train_img_dir = os.path.join(base_path, "images/train")
val_img_dir   = os.path.join(base_path, "images/val")
train_lbl_dir = os.path.join(base_path, "labels/train")
val_lbl_dir   = os.path.join(base_path, "labels/val")

In [ ]:
for d in [train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir]:
    os.makedirs(d, exist_ok=True)

folders = {
    "train_images": train_img_dir,
    "train_labels": train_lbl_dir,
    "val_images": val_img_dir,
    "val_labels": val_lbl_dir
}

for name, path in folders.items():
    num_files = len(os.listdir(path))
    print(f"{name}: {num_files} files")


train_images: 7096 files
train_labels: 7095 files
val_images: 1729 files
val_labels: 1729 files


In [ ]:
def fast_imread(path):
    try:
        img_bytes = np.fromfile(path, dtype=np.uint8)
        img = cv2.imdecode(img_bytes, cv2.IMREAD_COLOR)
        return img
    except:
        return None

def is_label_valid(label_path):
    if not os.path.exists(label_path):
        return False

    with open(label_path, "r") as f:
        lines = f.readlines()
        if len(lines) == 0:
            return False

        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5:
                return False
            try:
                cls, x, y, w, h = map(float, parts)
            except:
                return False
            if not (0 <= x <= 1 and 0 <= y <= 1 and 0 <= w <= 1 and 0 <= h <= 1):
                return False

    return True


In [ ]:
model = YOLO("yolov8n.pt")

results = model.train(
    data="/content/drive/My Drive/new grad data/sperm_dataset.yaml",
    imgsz=640,
    epochs=15,
    batch=16,
    patience=5,
    optimizer="Adam",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    dropout=0.0,
    # Augmentations
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=0.0,
    mixup=0.0,
    # Accelerate training
    device=0,
    workers=4,
    # Saving configs
    project="sperm_detection_results",
    name="yolov8s_sperm_best",
    save=True,
    save_period=2,
    exist_ok=True
)


Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/My Drive/new grad data/sperm_dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=yolov8s_sperm_best, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam, overlap_mask=True, pati

In [ ]:
import shutil

src = "/content/runs/detect/sperm_detection_results/yolov8s_sperm_best"
dst = "/content/drive/My Drive/new grad data/sperm_detection_results/yolov8n_sperm_best"

shutil.move(src, dst)


'/content/drive/My Drive/new grad data/sperm_detection_results/yolov8n_sperm_best'

In [ ]:
# Load model
model_path = "/content/drive/My Drive/new grad data/sperm_detection_results/yolov8n_sperm_best/weights/best.pt"
model = YOLO(model_path)

# Run validation
metrics = model.val(
    data="/content/drive/My Drive/new grad data/sperm_dataset.yaml",
    save_json=True
)

# Print a summary of metrics
metrics.summary()

# Optionally, access metrics as a dict
results = metrics.results_dict
print(results)


Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.2 ms, read: 22.7±6.1 MB/s, size: 43.9 KB)
val: Scanning /content/drive/My Drive/new grad data/labels/val.cache... 1729 images, 12 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1729/1729 290.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 109/109 2.7it/s 40.2s
                   all       1729      35335       0.97      0.933      0.963      0.682
                 sperm       1717      35335       0.97      0.933      0.963      0.682
Speed: 1.2ms preprocess, 3.4ms inference, 0.0ms loss, 1.9ms postprocess per image
Saving /content/runs/detect/val4/predictions.json...
Results saved to /content/runs/detect/val4
{'metrics/precision(B)': 0.970022769374648, 'metrics/recall(B)': 0.9334371020234895, 'metrics/mAP50(B)': 0.9630525795

In [ ]:
# ==== Paths ====
model_path = "/content/drive/My Drive/new grad data/best weights for detection.pt"
input_video = "/content/drive/My Drive/new grad data/test_videos/test data/68.mp4"
output_video = "/content/test_video_out.mp4"

# ==== Load model ====
model = YOLO(model_path)

# ==== Open video ====
cap = cv2.VideoCapture(input_video)
if not cap.isOpened():
    print("Error opening video file")

# Get video info
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Define VideoWriter
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

# ==== Process frames ====
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # YOLO inference (return annotated frame)
    results = model.predict(frame, imgsz=416)

    annotated_frame = results[0].plot()

    # Write frame to output video
    out.write(annotated_frame)

# ==== Release resources ====
cap.release()
out.release()
cv2.destroyAllWindows()

print("Done! Video saved at:", output_video)
